# Estimasi Harga Mobil Bekas (Preparation Model + Streamlit via Cloudflare Tunnel)
Notebook ini menggabungkan pembuatan model (`estimasi_mobil.sav`) sekaligus menjalankan aplikasi web **Streamlit** langsung dari Google Colab menggunakan **Cloudflare Tunnel**.

**Catatan Penggunaan:**
1. Upload file `toyota.csv` ke penyimpanan sesi Google Colab melalui menu **Files** (ikon folder di sebelah kiri).
2. Jalankan seluruh sel secara berurutan (**Runtime > Run all**).
3. Klik tautan `https://xxxx-xxxx.trycloudflare.com` yang muncul pada output sel paling bawah.

In [ ]:
# [OPSIONAL] Jalankan sel ini jika ingin upload file toyota.csv lewat tombol upload
# from google.colab import files
# uploaded = files.upload()

# 1. Deskripsi Library yang Digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Memanggil Dataset

In [ ]:
df = pd.read_csv('toyota.csv')

# 3. Deskripsi Dataset

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
sns.heatmap(df.isnull())

In [ ]:
df.describe()

# 4. Visualisasi Data

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True)

## Jumlah Mobil Berdasarkan Model

In [ ]:
models = df.groupby('model').count()[['tax']].sort_values(by='tax', ascending=True).reset_index()
models = models.rename(columns={'tax':'numberOfCars'})

fig = plt.figure(figsize=(15,5))
sns.barplot(x=models['model'], y=models['numberOfCars'], color='royalblue')
plt.xticks(rotation=60)

## Ukuran Mesin

In [ ]:
engine = df.groupby('engineSize').count()[['tax']].sort_values(by='tax').reset_index()
engine = engine.rename(columns={'tax':'count'})

plt.figure(figsize=(15,5))
sns.barplot(x=engine['engineSize'], y=engine['count'], color='royalblue')

## Distribusi Mileage & Harga Mobil

In [ ]:
plt.figure(figsize=(15,5))
sns.distplot(df['mileage'])

In [ ]:
plt.figure(figsize=(15,5))
sns.distplot(df['price'])

# 5. Seleksi Fitur & Split Data

In [ ]:
features = ['year', 'mileage', 'tax', 'mpg', 'engineSize']
x = df[features]
y = df['price']
x.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=70)
y_test.shape

# 6. Membuat Model Regresi Linier & Save Model

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(x_train, y_train)
pred = lr.predict(x_test)

score = lr.score(x_test, y_test)
print('Akurasi model Regresi Linier = ', score)

In [ ]:
# Uji Inputan Model
input_data = np.array([[2019, 5000, 145, 30.2, 2]])
prediction = lr.predict(input_data)
print('Estimasi harga mobil dalam EUR :', prediction)

In [ ]:
# Save Model ke estimasi_mobil.sav
import pickle

filename = 'estimasi_mobil.sav'
pickle.dump(lr, open(filename, 'wb'))

---
# 7. Deploy Aplikasi Streamlit di Google Colab (Cloudflare Tunnel)
Bagian di bawah ini membuat file `estimasi_mobil.py`, menginstal `streamlit` serta `cloudflared`, lalu menjalankan server Streamlit yang dihubungkan ke internet via Cloudflare Quick Tunnel.

In [ ]:
%%writefile estimasi_mobil.py
import pickle
import streamlit as st

model = pickle.load(open('estimasi_mobil.sav', 'rb'))

st.title('Estimasi Harga Mobil Bekas')

year = st.number_input('Input Tahun Mobil')
mileage = st.number_input('Input Km Mobil')
tax = st.number_input('Input Pajak Mobil')
mpg = st.number_input('Input Konsumsi BBM Mobil')
engineSize = st.number_input('Input Engine Size')

predict = ''

if st.button('Estimasi Harga'):
    predict = model.predict(
        [[year, mileage, tax, mpg, engineSize]]
    )
    st.write('Estimasi harga mobil bekas dalam Ponds : ', predict)
    st.write('Estimasi harga mobil bekas dalam IDR (Juta) :', predict*19000)

In [ ]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!streamlit run estimasi_mobil.py &>/content/logs.txt &
!cloudflared tunnel --url http://localhost:8501